# 🚕 04 — Full-year retrain with hourly weather

**Run this on your machine** (it needs internet for the TLC files and Open-Meteo, and several GB of RAM). It fixes three problems in notebook 02 and re-measures everything:

1. **Weather covered 38 days.** `dim_clima.csv` only has the first 3–4 days of each month, and notebook 02 drops every trip without weather, so the "full-year" model trained on 3.95M trips from 38 days. Here hourly 2024 weather comes from Open-Meteo's historical archive (the same provider and units the gateway uses live) and is joined on the pickup hour, so no trip is lost to the weather join and training weather matches serving weather.
2. **Route encoding saw each trip's own fare.** Training rows now get out-of-fold route means (scikit-learn `TargetEncoder`), so a trip's fare never feeds its own feature.
3. **Tuning was not time-ordered.** The 500K tuning sample is seeded and kept in time order, so `TimeSeriesSplit` really validates on later trips.

Then it scores the holdout against a linear baseline and the meter formula, and again with **request-time inputs** (the training split's median distance and duration for the same route and time of day, instead of the recorded values). Finally it exports the model and `route_means.json` for the API and prints a README-ready table.

In [ ]:
# Settings
SAMPLE_FRACTION = 0.25   # share of each month's filtered trips to keep (1.0 = all, needs a lot of RAM)
MONTHS = range(1, 13)
RANDOM_STATE = 42
TUNING_SAMPLE = 500_000

In [ ]:
import json
import gc
import urllib.request
import numpy as np
import pandas as pd
import xgboost as xgb
import joblib
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score
from sklearn.model_selection import KFold, RandomizedSearchCV, TimeSeriesSplit
from sklearn.preprocessing import TargetEncoder

if Path('service/ml-inference/data').exists():
    DATA_DIR = Path('service/ml-inference/data')
elif Path('../service/ml-inference/data').exists():
    DATA_DIR = Path('../service/ml-inference/data')
else:
    raise FileNotFoundError("Run this notebook from the project root or the notebooks folder.")
EXPORT_DIR = DATA_DIR.parent   # service/ml-inference: where the API looks for the model

def download_month(month):
    path = DATA_DIR / f"yellow_tripdata_2024-{month:02d}.parquet"
    if not path.exists():
        print(f"Downloading {path.name} (~50 MB)...")
        urllib.request.urlretrieve(f"https://d37ci6vzurychx.cloudfront.net/trip-data/{path.name}", path)
    return path

parquet_files = [download_month(m) for m in MONTHS]
print(f"{len(parquet_files)} monthly files ready")

## 1. Hourly 2024 weather from Open-Meteo

Same variables and units as the gateway's live call: temperature (°C), precipitation (mm in the past hour), snowfall (cm). Saved to `weather_hourly_2024.csv` so later runs skip the download.

In [ ]:
WEATHER_CSV = DATA_DIR / "weather_hourly_2024.csv"
if not WEATHER_CSV.exists():
    url = ("https://archive-api.open-meteo.com/v1/archive?latitude=40.7128&longitude=-74.0060"
           "&start_date=2024-01-01&end_date=2024-12-31"
           "&hourly=temperature_2m,precipitation,snowfall&timezone=America%2FNew_York")
    with urllib.request.urlopen(url, timeout=120) as response:
        hourly = json.load(response)["hourly"]
    pd.DataFrame({"hour": hourly["time"], "temp": hourly["temperature_2m"],
                  "precip": hourly["precipitation"], "snow": hourly["snowfall"]}).to_csv(WEATHER_CSV, index=False)

weather = pd.read_csv(WEATHER_CSV)
weather["hour"] = pd.to_datetime(weather["hour"]).astype("datetime64[ns]")
weather = weather.drop_duplicates("hour").sort_values("hour").ffill()
weather[["temp", "precip", "snow"]] = weather[["temp", "precip", "snow"]].astype("float32")
print(f"{len(weather):,} hourly rows, {weather.hour.min()} -> {weather.hour.max()}")
weather.describe().round(2)

## 2. Process each month (notebook 02's filters, weather joined on the pickup hour)

In [ ]:
zones = pd.read_csv(DATA_DIR / "dim_localizacao.csv").rename(columns={"localizacao_id": "LocationID", "bairro": "Borough"})[["LocationID", "Borough"]]

def process_month(path):
    cols = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance", "fare_amount", "PULocationID", "DOLocationID"]
    df = pd.read_parquet(path, columns=cols)
    df = df[(df.tpep_pickup_datetime >= "2024-01-01") & (df.tpep_pickup_datetime < "2025-01-01")]
    df["trip_time"] = (df.tpep_dropoff_datetime - df.tpep_pickup_datetime).dt.total_seconds()
    df = df.rename(columns={"trip_distance": "trip_miles"})
    df = df[(df.trip_miles > 0.1) & (df.trip_miles < 100) & (df.fare_amount > 2.50) & (df.fare_amount < 400) & (df.trip_time > 60)]
    if SAMPLE_FRACTION < 1:
        df = df.sample(frac=SAMPLE_FRACTION, random_state=RANDOM_STATE)
    df = df.merge(zones.rename(columns={"LocationID": "PULocationID", "Borough": "PU_Borough"}), on="PULocationID")
    df = df.merge(zones.rename(columns={"LocationID": "DOLocationID", "Borough": "DO_Borough"}), on="DOLocationID")

    df["tpep_pickup_datetime"] = df["tpep_pickup_datetime"].astype("datetime64[ns]")
    df = df.sort_values("tpep_pickup_datetime")
    df = pd.merge_asof(df, weather, left_on="tpep_pickup_datetime", right_on="hour", direction="backward")

    df["request_hour"] = df.tpep_pickup_datetime.dt.hour.astype("int8")
    df["is_weekend"] = (df.tpep_pickup_datetime.dt.dayofweek >= 5).astype("int8")
    df["is_airport_trip"] = (df.PULocationID.isin([132, 138, 1]) | df.DOLocationID.isin([132, 138, 1])).astype("int8")
    df["implied_speed_mph"] = (df.trip_miles / df.trip_time * 3600).clip(0, 75).astype("float32")
    df["route_id"] = (df.PULocationID * 1000 + df.DOLocationID).astype("int32")
    df = pd.get_dummies(df, columns=["PU_Borough", "DO_Borough"], dtype="uint8")
    df["fare_amount"] = df["fare_amount"].astype("float32")
    df["trip_miles"] = df["trip_miles"].astype("float32")
    df["trip_time"] = df["trip_time"].astype("float32")
    return df.drop(columns=["tpep_dropoff_datetime", "hour"])

chunks = []
for path in parquet_files:
    chunks.append(process_month(path))
    print(f"   {path.name}: {len(chunks[-1]):,} trips")
trips = pd.concat(chunks, ignore_index=True)
del chunks; gc.collect()
dummy_cols = [c for c in trips.columns if c.startswith(("PU_Borough_", "DO_Borough_"))]
trips[dummy_cols] = trips[dummy_cols].fillna(0).astype("uint8")   # a borough missing from one month's sample
trips = trips.sort_values("tpep_pickup_datetime").reset_index(drop=True)
print(f"Total: {len(trips):,} trips on {trips.tpep_pickup_datetime.dt.normalize().nunique()} distinct days "
      f"(notebook 02: 3,949,995 trips on 38 days); missing weather: {trips.temp.isna().sum():,}")

## 3. Chronological split and out-of-fold route encoding

In [ ]:
split = int(len(trips) * 0.8)
train, test = trips.iloc[:split].copy(), trips.iloc[split:].copy()
del trips; gc.collect()
print(f"Train {len(train):,} ({train.tpep_pickup_datetime.min():%Y-%m-%d} -> {train.tpep_pickup_datetime.max():%Y-%m-%d}) | "
      f"test {len(test):,} ({test.tpep_pickup_datetime.min():%Y-%m-%d} -> {test.tpep_pickup_datetime.max():%Y-%m-%d})")

# Out-of-fold for training rows (no trip sees its own fare); full-train means for the test set and the API.
encoder = TargetEncoder(target_type="continuous", smooth=50.0, cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE))
train["historical_route_fare"] = encoder.fit_transform(train[["route_id"]], train["fare_amount"]).ravel().astype("float32")
test["historical_route_fare"] = encoder.transform(test[["route_id"]]).ravel().astype("float32")
route_means = {str(int(r)): float(v) for r, v in zip(encoder.categories_[0], encoder.encodings_[0])}
print(f"{len(route_means):,} routes encoded")

borough_cols = sorted(c for c in train.columns if c.startswith(("PU_Borough_", "DO_Borough_")))
FEATURES = ["trip_miles", "trip_time", "request_hour", "is_weekend", "is_airport_trip",
            "temp", "precip", "snow", "implied_speed_mph"] + borough_cols + ["historical_route_fare"]
X_train, y_train = train[FEATURES], train["fare_amount"]
X_test, y_test = test[FEATURES], test["fare_amount"]
print(len(FEATURES), "features:", FEATURES)

## 4. Tune on a seeded, time-ordered sample, then train on the full training split

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
idx = np.sort(rng.choice(len(X_train), size=min(TUNING_SAMPLE, len(X_train)), replace=False))  # sorted = still in time order
param_grid = {
    "n_estimators": [200, 300, 400],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [6, 8, 10],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.7, 0.8, 0.9],
}
search = RandomizedSearchCV(
    xgb.XGBRegressor(tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=param_grid, n_iter=10, scoring="neg_mean_absolute_error",
    cv=TimeSeriesSplit(n_splits=3), random_state=RANDOM_STATE, n_jobs=1, verbose=1,
)
search.fit(X_train.iloc[idx], y_train.iloc[idx])
best_params = search.best_params_
print("Best parameters:", best_params)

model = xgb.XGBRegressor(**best_params, tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1)
model.fit(X_train, y_train)
print("Trained on", f"{len(X_train):,}", "trips")

## 5. Holdout: XGBoost vs baselines, and with request-time inputs

Request-time inputs: each test trip's distance and duration are replaced by the training split's median for the same route and 3-hour time-of-day bucket (falling back to the route median). That is roughly what a lookup-based estimate would give a live request; trips on routes never seen in training are left out of that row and the coverage is printed.

In [ ]:
results = []
def score(name, y_true, pred):
    results.append({"Method": name, "Trips": len(y_true),
                    "MAE ($)": round(mean_absolute_error(y_true, pred), 2),
                    "RMSE ($)": round(float(np.sqrt(mean_squared_error(y_true, pred))), 2),
                    "MAPE (%)": round(100 * mean_absolute_percentage_error(y_true, pred), 2),
                    "R²": round(r2_score(y_true, pred), 3)})

pred = model.predict(X_test)
score("XGBoost (recorded distance and time)", y_test, pred)

lin_cols = ["trip_miles", "trip_time", "is_airport_trip"]
linear = LinearRegression().fit(train[lin_cols], y_train)
score("Linear regression (miles, seconds, airport flag)", y_test, linear.predict(test[lin_cols]))
meter = np.maximum(3.0, 3.0 + 0.7 * np.maximum(5 * test.trip_miles, test.trip_time / 60))
score("Meter formula: 3 + 0.70 × max(5 × miles, minutes)", y_test, meter)

# Request-time inputs from the training split's medians
train["bucket"] = train.request_hour // 3
test["bucket"] = test.request_hour // 3
by_route_bucket = train.groupby(["route_id", "bucket"])[["trip_miles", "trip_time"]].median()
by_route = train.groupby("route_id")[["trip_miles", "trip_time"]].median()
est = test[["route_id", "bucket"]].join(by_route_bucket, on=["route_id", "bucket"])
fallback = test[["route_id"]].join(by_route, on="route_id")
est["trip_miles"] = est["trip_miles"].fillna(fallback["trip_miles"])
est["trip_time"] = est["trip_time"].fillna(fallback["trip_time"])
covered = est["trip_miles"].notna().to_numpy()
X_est = X_test[covered].copy()
X_est["trip_miles"] = est.loc[covered, "trip_miles"].astype("float32")
X_est["trip_time"] = est.loc[covered, "trip_time"].astype("float32")
X_est["implied_speed_mph"] = (X_est.trip_miles / X_est.trip_time * 3600).clip(0, 75).astype("float32")
print(f"Request-time estimates available for {covered.mean():.1%} of test trips")
score("XGBoost, same trips, recorded inputs", y_test[covered], pred[covered])
score("XGBoost, same trips, request-time estimates", y_test[covered], model.predict(X_est))

summary = pd.DataFrame(results)
summary

## 6. Export for the API and a README-ready table

In [ ]:
joblib.dump(model, EXPORT_DIR / "production_taxi_master_model_2024.joblib")
with open(EXPORT_DIR / "route_means.json", "w") as f:
    json.dump(route_means, f)
print("Saved model and route_means.json to", EXPORT_DIR)

lin_mae = summary.loc[summary.Method.str.startswith("Linear"), "MAE ($)"].item()
xgb_mae = summary.loc[summary.Method == "XGBoost (recorded distance and time)", "MAE ($)"].item()
print(f"\nXGBoost cuts holdout MAE by {1 - xgb_mae / lin_mae:.0%} versus the linear baseline.\n")
print("| " + " | ".join(summary.columns) + " |")
print("|" + "---|" * len(summary.columns))
for _, row in summary.iterrows():
    print("| " + " | ".join(f"{v:,}" if isinstance(v, (int, np.integer)) else str(v) for v in row) + " |")